# EAGLE Steering Pipeline: Iterative Generation via Latent Space Rewards

This notebook demonstrates a toy example pipeline where a generative language model is iteratively steered and judged by an embedding model. The embedding model (ELM) acting as the reward signal is `BAAI/bge-m3`, and the generative agent is `Qwen2.5-0.5B-Instruct`.

Unlike traditional prompting where an LLM is given a single chance to generate an answer, this pipeline treats text generation as a Reinforcement Learning (RL) problem. The agent iteratively explores different ways to rewrite its own draft, guided by a mathematical reward function.

## Pipeline Architecture

This notebook is broken down into four core phases:

1. **Contrastive Fine-Tuning:** We fine-tune the `bge-m3` embedding model using `MultipleNegativesRankingLoss`. This teaches the embedding space to understand the semantic mapping between mechanical actions (hypotheses) and structural outcomes (conclusions).
2. **Baseline Generation:** The generative LLM produces a zero-shot, unsteered answer to a prompt. This serves as our starting state ($S_0$).
3. **Dynamic Action Space (Brainstorming):** Instead of using hardcoded edits, the LLM acts as an autonomous agent to evaluate its current draft and brainstorm context-specific instructions (actions) to improve it.
4. **The Iterative Optimization Loop:** The agent applies its brainstormed instructions to generate candidate drafts. The ELM evaluates these drafts in latent space using cosine similarity against a target vector. 

In [3]:
### Import Block
import pandas as pd
from datasets import Dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, losses
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, losses
import random
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

ModuleNotFoundError: No module named 'datasets'

## 1. Dataset Initialization
To train a contrastive embedding model, we establish a dataset of paired statements.

Think of this dataset like a logbook of cause and effect. If the hypothesis is the mechanical action (like pressing the gas pedal), the conclusion is the observed result (the car accelerates). We are preparing this data so the embedding model learns to map these specific technical actions closely to their structural outcomes in latent space.

In [ ]:


mars_toy_data = {
    "hypothesis": [
        "Why does the fourth planet from the sun appear to have a reddish hue?",
        "What geological composition is responsible for the distinct color of Mars?",
        "Could you explain the reason behind the Martian surface looking like rust?",
        "I am looking at Mars through a telescope; what causes that red tint?"
    ],
    "conclusion": [
        "Mars appears red because its surface is covered in iron oxide, commonly known as rust.",
        "The distinct reddish color of the planet is a direct result of abundant iron oxide in its regolith.",
        "The rusty appearance is caused by the oxidation of iron-rich minerals across the Martian landscape.",
        "That red tint is due to a thick layer of iron oxide dust covering the planet's surface."
    ]
}

df_mars = pd.DataFrame(mars_toy_data)

# Rename columns for MultipleNegativesRankingLoss compatibility
df_mars = df_mars.rename(columns={"hypothesis": "anchor", "conclusion": "positive"})
mars_dataset = Dataset.from_pandas(df_mars)

print(f"Dataset ready. Total training pairs: {len(mars_dataset)}")


## 2. Bi-Encoder Fine-Tuning
This section initializes the embedding model and executes the contrastive learning loop. The column names must be mapped to anchor and positive before passing the dataset to the trainer.

Component Documentation

`MultipleNegativesRankingLoss`: A contrastive loss function that expects pairs of related text.

Concept Analogy: Think of it like organizing a seating chart at a wedding. The loss function pulls the matched pair (the anchor and positive) to the same table while actively pushing all other mismatched guests in the room (the in-batch negatives) as far away as possible. We want to caputre the logical relationship of hypothesis to correct conclusions. 

`per_device_train_batch_size`: Set to 2 to prevent Out-Of-Memory (OOM) errors during local prototyping. In contrastive learning, larger batch sizes generally yield better embeddings because the model sees more "negative" examples per step, but this requires more VRAM. For the toy pipeline this is only nessarcary if running locally (preferred Google Collab.)

In [ ]:


# 1. Load BGE-M3 (or bge-small-en-v1.5 for ultra-fast local prototyping)
embedding_model = SentenceTransformer("BAAI/bge-m3")

# 2. Define the contrastive loss function
loss = losses.MultipleNegativesRankingLoss(embedding_model)

# 3. Setup lightweight training args
args = SentenceTransformerTrainingArguments(
    output_dir="./bge_m3_toy_checkpoint",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    learning_rate=2e-5,
    logging_steps=1,
    save_strategy="no"
)

trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=mars_dataset,
    loss=loss
)

trainer.train()

## 3. Generative Agent Setup & Target Vectorization
This section initializes the primary LLM (the Agent) that will attempt to generate and iteratively refine text. It also defines the target state and converts it into a mathematical vector using the fine-tuned embedding model.

Component Documentation

target_vec: The continuous vector representation of the desired output.

Concept Analogy: Think of target_vec as a specific GPS coordinate for a destination. The generative model is a driver trying to navigate there. Instead of rigidly checking if the driver's current street name exactly matches the destination (exact string matching), the embedding model calculates the straight-line physical distance (cosine similarity) between the driver's current location and the target coordinate. This allows the system to recognize when the model is getting "warmer" or "colder" during the steering process

In [ ]:


device = "cuda" if torch.cuda.is_available() else "cpu"

# =====================================================================
# 1. SETUP: MODELS & SYNTHETIC DATA
# =====================================================================

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
llm = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32).to(device)

hypothesis = "How large is mars?"
target_conclusion = "Mars is the second-smallest planet in the Solar System, with a radius of approximately 3,389 kilometers, about half the size of Earth."
target_vec = embedding_model.encode(target_conclusion, convert_to_tensor=True)

## 4. Baseline Generation & Initial Evaluation
This section establishes the starting state. The LLM generates a standard, zero-shot response to the input question. We then evaluate how semantically close this initial draft is to our target conclusion, setting the baseline utility score before any iterative steering occurs.

Component Documentation

ChatML Template (<|im_start|>): The native structural tagging the Qwen model was fine-tuned on. Proper formatting ensures the model understands role assignments (system, user, assistant) rather than treating the prompt as a continuous string of raw text.

`F.cosine_similarity`: Measures the angular distance between two vectors in the embedding space.

Concept Analogy: Imagine two arrows originating from the center of a room. If they point in the exact same direction, their similarity is 1.0 (perfect semantic match). If they are perpendicular, the similarity is 0. If they point in completely opposite directions, it's -1.0. This metric allows us to mathematically score how close the LLM's current draft is to the target meaning, regardless of the specific vocabulary used.

In [ ]:

# =====================================================================
# 2. BASELINE GENERATION
# =====================================================================
print("=== BASELINE (UNSTEERED) GENERATION ===")

baseline_prompt = (
    f"<|im_start|>system\nYou are a question answering machine. Answer the question in detail.<|im_end|>\n"
    f"<|im_start|>user\nQuestion:\nText: \"{hypothesis}\"<|im_end|>\n"
    f"<|im_start|>assistant\n"
)
inputs = tokenizer(baseline_prompt, return_tensors="pt").to(device)

with torch.no_grad():
    unsteered_out = llm.generate(**inputs, max_new_tokens=40, do_sample=False)

baseline_text = tokenizer.decode(unsteered_out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
baseline_vec = embedding_model.encode(baseline_text, convert_to_tensor=True)
baseline_sim = F.cosine_similarity(baseline_vec.unsqueeze(0), target_vec.unsqueeze(0)).item()

print(f"Baseline Text: {baseline_text}")
print(f"Baseline Cosine Similarity: {baseline_sim:.4f}\n")


## 5. Action Space Generation (Agent Brainstorming)

This section defines the dynamic action space ($\mathcal{A}_t$) for the reinforcement learning loop. Instead of relying on a hardcoded list of editing instructions, the LLM acts as an agent that evaluates the current state (the draft) and brainstorms multiple potential ways to rewrite it.
Component Documentation

`failure_context`: A contextual flag injected into the prompt when the loop stalls. It prevents the model from blindly suggesting the same failing instructions repeatedly.

`dynamic_temp ` (Temperature Scaling): Dynamically increases the temperature parameter of the LLM based on how long the system has been stalled.

Concept Analogy: Think of temperature like a thermostat for a brainstorming room. At a low temperature (0.6), the ideas are highly predictable and focused. However, if the team gets stuck on a bad idea (stalled_count > 0), we crank up the heat (up to 1.3). This forces the room to become more chaotic and generate wildly different, unpredictable ideas, helping the system break out of its creative rut (escaping a local minimum).

Parsing Logic extracts the numbered list from the unstructured LLM output into a clean Python list of discrete actions.

In [ ]:
# =====================================================================
# 3. ACTION SPACE (A): AGENT BRAINSTORMING REASONING BLOCK
# =====================================================================
def brainstorm_candidate_actions(current_state: str, stalled_count: int) -> list[str]:
    """
    The EAGLE Agent (\pi) brainstorms a state-dependent set of actions (A_t).
    """
    # FIX 4: Inject failure signal into the prompt if the state is stagnating
    failure_context = ""
    if stalled_count > 0:
        failure_context = "\nNote: Previous instructions failed to improve the draft. Suggest radically different, creative rewriting angles."

    # FIX 3: Apply model-native ChatML template
    brainstorm_prompt = (
        f"<|im_start|>user\nReview this draft: '{current_state}'\n"
        f"Brainstorm 3 short, distinct instructional prompts to rewrite this draft "
        f"so it logically acts as a response to the question '{hypothesis}'.{failure_context}\n"
        f"Instructions:\n1.<|im_end|>\n"
        f"<|im_start|>assistant\n1. "
    )
    inputs = tokenizer(brainstorm_prompt, return_tensors="pt").to(device)
    
    # FIX 4: Dynamically scale temperature to escape local minima
    dynamic_temp = min(0.6 + (stalled_count * 0.2), 1.3)
    
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=60, do_sample=True, temperature=dynamic_temp)
        
    generated_text = "1. " + tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    actions = []
    for line in generated_text.split('\n'):
        line = line.strip()
        if line and line[0].isdigit():
            actions.append(line.split('.', 1)[-1].strip())
            
    if not actions:
        actions = ["Summarize this statement"]
        
    return actions[:3]

## 6. The Iterative Reinforcement Learning LoopThis is the execution core of the EAGLE pipeline.

 The system iterates through proposing actions, drafting new states, and evaluating them in latent space using the embedding model as a reward signal.
 
 Component Documentation
 
 `Steering Prompt` (The Environment / $\mathcal{P}$): This prompt applies the brainstormed action to the current draft.
 
 Concept Analogy: If the brainstormed action is the architectural blueprint, the steering prompt is the construction crew actually rebuilding the house. We must pass them both the blueprint and the original foundation (the hypothesis) so they don't lose sight of the core structure.
 
 `Reward Function` ($r$): Calculates the change in cosine similarity between the candidate draft and the target conclusion.Concept Analogy: Think of the reward function like a metal detector. The embedding model acts as the sensor, emitting a stronger signal (higher utility score) the closer the LLM's generated text physically moves toward the target meaning in latent space.
 
 `State Update Logic` (Simulated Annealing): The use of tolerance and random.random() to accept slightly worse states.
 
 Concept Analogy: Imagine a hiker trying to find the highest peak in a mountain range while blindfolded. If they refuse to ever take a step downhill (strict greedy search), they will get stuck on the top of the first small hill they find. By occasionally allowing a step downhill (accepting a slight degradation in utility), the hiker can cross a valley to eventually climb a much taller mountain (escaping a local minimum).

In [2]:
# =====================================================================
# 4. EAGLE PIPELINE: ITERATIVE RL LOOP 
# =====================================================================
print("=== EAGLE STEERED GENERATION ===")
current_state_text = baseline_text
current_utility = baseline_sim

global_best_text = baseline_text
global_best_utility = baseline_sim
stalled_count = 0 # Track stagnation

for step in range(1, 15):
    # 1. Agent dynamically generates context-specific actions based on the current state
    candidate_actions = brainstorm_candidate_actions(current_state_text, stalled_count)
    
    best_action = None
    best_next_state = None
    best_utility = -float("inf")

    # 2. Evaluate candidate actions in the Environment (P)
    for action in candidate_actions:
        # FIX 3 & 5: Apply ChatML template and anchor the Original Question
        steer_prompt = (
            f"<|im_start|>user\nOriginal Question: {hypothesis}\n"
            f"Current Draft: {current_state_text}\n"
            f"Instruction: {action}\n"
            f"Rewrite the draft based on the instruction to better answer the Original Question.<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
        
        steer_inputs = tokenizer(steer_prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            steered_out = llm.generate(**steer_inputs, max_new_tokens=40, do_sample=False)
            
        next_state_candidate = tokenizer.decode(steered_out[0][steer_inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        
        # 3. Reward Function (r): Evaluate mathematically in Latent Space (Z)
        candidate_vec = embedding_model.encode(next_state_candidate, convert_to_tensor=True)
        utility_score = torch.nn.functional.cosine_similarity(candidate_vec.unsqueeze(0), target_vec.unsqueeze(0)).item()
        
        if utility_score > best_utility:
            best_utility = utility_score
            best_action = action
            best_next_state = next_state_candidate

    reward = best_utility - current_utility
    
    print(f"--- STEP {step} ---")
    print("Brainstormed Actions:")
    for idx, act in enumerate(candidate_actions):
        print(f"  - {act}")
    print(f"\nSelected Action (a_t): '{best_action}'")
    print(f"New State (s_t+1): '{best_next_state}'")
    print(f"New Utility: {best_utility:.4f} (Reward: {reward:+.4f})\n")

    # State update logic

    # Inside your step loop:
    tolerance = 0.05  # Allow up to a 5% drop in similarity

    # Accept if it's an improvement, OR occasionally accept a slight degradation to explore
    if reward > 0 or (reward > -tolerance and random.random() < 0.2):
        current_state_text = best_next_state
        current_utility = best_utility
        stalled_count = 0
        
        # Still only save the absolute best to the global tracker
        if best_utility > global_best_utility:
            global_best_text = best_next_state
            global_best_utility = best_utility
    else:
        stalled_count += 1
        print(f"Notice: Utility degraded. State held. (Stall Count: {stalled_count})\n")

print("=== FINAL OPTIMIZED OUTPUT ===")
print(f"Final Steered Text: '{global_best_text}'")
print(f"Final Utility: {global_best_utility:.4f}")

=== EAGLE STEERED GENERATION ===


NameError: name 'baseline_text' is not defined